# 02 - Layer-wise activation shift heatmaps (Q2)

Per (victim, condition, fact_type) cell, plot turn x layer mean cosine shift,
and the CoT - bare contrast. Localizes WHERE in the network the
representation moves under each attack type.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.io import load_jsonl
from src.analysis.shift import shift_table, heatmap_matrix, condition_contrast

In [ ]:
RUNS_DIR = pathlib.Path('../../runs')
rows = []
for ef in RUNS_DIR.rglob('exchanges.jsonl'):
    ex = load_jsonl(ef)
    rows.append(shift_table(ex, ef))
df = pd.concat(rows, ignore_index=True)
df.shape

In [ ]:
for victim in df['victim'].unique():
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, cond in zip(axes[:2], ['bare', 'cot']):
        m = heatmap_matrix(df, victim=victim, condition=cond)
        sns.heatmap(m, ax=ax, cmap='viridis', cbar_kws={'label': 'cosine shift'})
        ax.set_title(f'{victim} / {cond}')
    c = condition_contrast(df, victim=victim)
    sns.heatmap(c, ax=axes[2], cmap='RdBu_r', center=0,
                cbar_kws={'label': 'CoT - bare'})
    axes[2].set_title(f'{victim}: CoT - bare')
    plt.tight_layout()
    plt.show()

## Reading the contrast plot
Red cells = CoT-backed attack induces a *larger* representational shift than bare denial at that (turn, layer); blue = the reverse. Strong red bands at mid-to-late layers under CoT-backed attacks would support the hypothesis that fabricated CoT reaches deeper into the model's compute graph than flat denial.